# Unit 2 | NLP 03 — Text Classification & Model Comparison

This notebook builds and compares text classification pipelines across every vectorization approach:

1. **Feature preparation** — CountVec, TF-IDF, and Sentence Embeddings
2. **Systematic model grid** — every classifier × every representation, ranked by Macro F1
3. **Results visualization** — accuracy, F1, and fit/predict timing
4. **Error analysis** — where classical and embedding models disagree
5. **Grid Search** — k-fold cross-validation for TF-IDF + Logistic Regression
6. **Explainability** — RF feature importance, LR coefficients, and SHAP

**Dataset:** sklearn 20 Newsgroups

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.base import clone
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)
from sentence_transformers import SentenceTransformer
import shap

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
plt.style.use('dark_background')

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
CATEGORIES   = ['sci.space', 'comp.graphics', 'rec.sport.hockey', 'talk.religion.misc']
RANDOM_STATE = 42
MAX_FEATURES = 10000
EMB_MODEL    = 'all-MiniLM-L6-v2'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
train = fetch_20newsgroups(
    subset='train', categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'), random_state=RANDOM_STATE
)
test = fetch_20newsgroups(
    subset='test', categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'), random_state=RANDOM_STATE
)

print(f"Train: {len(train.data):,} docs   Test: {len(test.data):,} docs")
print(f"Classes: {train.target_names}")

## Section 1: Prepare All Feature Sets

We compute all representations up front so that model timing reflects only fit+predict, not vectorization.

- **CountVec / TF-IDF**: sparse term matrices (`MAX_FEATURES=10000`)
- **tfidf_shap**: smaller TF-IDF at 5000 features used for SHAP (densifying 10k features requires ~172 MB)
- **Embeddings**: dense 384-dim vectors from `all-MiniLM-L6-v2` (download once, cached after)

The key rule: all vectorizers are **fit on training data only**, then transform both splits.

In [ ]:
# ── Classical vectorizers ────────────────────────────────────────────────────
print('Fitting CountVectorizer...', end=' ', flush=True)
cv = CountVectorizer(max_features=MAX_FEATURES, stop_words='english')
X_train_cv  = cv.fit_transform(train.data)
X_test_cv   = cv.transform(test.data)
print('done')

print('Fitting TfidfVectorizer...', end=' ', flush=True)
tfidf = TfidfVectorizer(max_features=MAX_FEATURES, stop_words='english', sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train.data)
X_test_tfidf  = tfidf.transform(test.data)
print('done')

# Smaller TF-IDF for SHAP (5k features keeps dense arrays manageable)
print('Fitting TfidfVectorizer (SHAP)...', end=' ', flush=True)
tfidf_shap = TfidfVectorizer(max_features=5000, stop_words='english', sublinear_tf=True)
X_train_tfidf_shap = tfidf_shap.fit_transform(train.data)
X_test_tfidf_shap  = tfidf_shap.transform(test.data)
print('done')

# ── Sentence embeddings ──────────────────────────────────────────────────────
print(f'Encoding with {EMB_MODEL}...')
emb_model   = SentenceTransformer(EMB_MODEL)
X_train_emb = emb_model.encode(train.data, show_progress_bar=True, batch_size=64)
X_test_emb  = emb_model.encode(test.data,  show_progress_bar=True, batch_size=64)
print(f'Embeddings: train={X_train_emb.shape}  test={X_test_emb.shape}')

## Section 2: Systematic Model Grid — All Classifiers × All Representations

We evaluate every valid combination of representation and classifier:

| Representation | Classifiers |
|---|---|
| CountVec | NaiveBayes, LogisticReg, LinearSVC, RandomForest, SGD |
| TF-IDF | NaiveBayes, LogisticReg, LinearSVC, RandomForest, SGD |
| Embeddings | LogisticReg, LinearSVC, RandomForest, SGD |

**Note:** MultinomialNB is excluded for embeddings — it requires non-negative features.
Each fit is timed (fit + predict seconds) for a direct speed comparison.

In [ ]:
def evaluate(name, clf, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    t_fit = time.time() - t0

    t0 = time.time()
    y_pred = clf.predict(X_te)
    t_pred = time.time() - t0

    return {
        'Model'       : name,
        'Accuracy'    : round(accuracy_score(y_te, y_pred), 4),
        'Macro F1'    : round(f1_score(y_te, y_pred, average='macro'), 4),
        'Fit (s)'     : round(t_fit, 2),
        'Predict (s)' : round(t_pred, 3),
        '_preds'      : y_pred,
        '_clf'        : clf,
        'Representation': '',  # filled below
        'Classifier'    : '',
    }

feature_sets = {
    'CountVec'  : (X_train_cv,    X_test_cv),
    'TF-IDF'    : (X_train_tfidf, X_test_tfidf),
    'Embeddings': (X_train_emb,   X_test_emb),
}

classifier_configs = [
    ('NaiveBayes',          True,  MultinomialNB()),
    ('LogisticReg(C=1)',    False, LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)),
    ('LinearSVC(C=1)',      False, LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
    ('RandomForest(n=200)', False, RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
    ('SGD(log_loss)',       False, SGDClassifier(loss='log_loss', learning_rate='optimal',
                                                 max_iter=100, random_state=RANDOM_STATE, n_jobs=-1)),
]

rows = []
for rep_name, (X_tr, X_te) in feature_sets.items():
    for clf_label, nb_only, clf_proto in classifier_configs:
        if nb_only and rep_name == 'Embeddings':
            continue   # MultinomialNB needs non-negative features
        name = f'{rep_name} + {clf_label}'
        print(f'  {name}...', end=' ', flush=True)
        row = evaluate(name, clone(clf_proto), X_tr, train.target, X_te, test.target)
        row['Representation'] = rep_name
        row['Classifier']     = clf_label
        print(f"Acc={row['Accuracy']:.4f}  F1={row['Macro F1']:.4f}")
        rows.append(row)

results_df = (
    pd.DataFrame(rows)
    .drop(columns=['_preds', '_clf'])
    .sort_values('Macro F1', ascending=False)
    .reset_index(drop=True)
)
results_df

In [ ]:
print('Macro F1 by Representation')
print('=' * 50)
print(pd.DataFrame(rows).groupby('Representation')['Macro F1'].describe().round(4))
print()
print('Macro F1 by Classifier')
print('=' * 50)
print(pd.DataFrame(rows).groupby('Classifier')['Macro F1'].describe().round(4))

In [ ]:
best_row   = max(rows, key=lambda r: r['Macro F1'])
best_name  = best_row['Model']
best_preds = best_row['_preds']
short_names = [c.split('.')[-1] for c in train.target_names]

print(f'Classification Report -- {best_name}')
print('=' * 60)
print(classification_report(test.target, best_preds, target_names=train.target_names))

cm = confusion_matrix(test.target, best_preds)
fig, ax = plt.subplots(figsize=(4, 4))
sns.heatmap(cm, annot=True, square=True, fmt='d',
            xticklabels=short_names, yticklabels=short_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix -- Best Model')
plt.tight_layout()
plt.show()

## Section 3: Results Visualization

In [ ]:
metrics = ['Accuracy', 'Macro F1']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, metrics):
    sorted_df = results_df.sort_values(metric, ascending=True)
    colors = []
    for name in sorted_df['Model']:
        if 'Embedding' in name:
            colors.append('#f1a65b')
        elif 'TF-IDF' in name:
            colors.append('#5b8dd9')
        else:
            colors.append('#9a5bd9')

    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=colors)
    ax.set_xlabel(metric)
    ax.set_title(metric)
    ax.set_xlim(sorted_df[metric].min() - 0.05, 1.0)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)

legend_elements = [
    Patch(facecolor='#9a5bd9', label='CountVec'),
    Patch(facecolor='#5b8dd9', label='TF-IDF'),
    Patch(facecolor='#f1a65b', label='Embeddings'),
]
axes[1].legend(handles=legend_elements, loc='lower right')
plt.suptitle('Model Comparison: Classical vs. Semantic', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
speed_df = results_df[['Model', 'Fit (s)', 'Predict (s)']].copy()
speed_df['Total (s)'] = speed_df['Fit (s)'] + speed_df['Predict (s)']
speed_df = speed_df.sort_values('Total (s)', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
x     = range(len(speed_df))
width = 0.4
ax.bar([i - width/2 for i in x], speed_df['Fit (s)'],     width, label='Fit',     color='#5b8dd9')
ax.bar([i + width/2 for i in x], speed_df['Predict (s)'], width, label='Predict', color='#5bd95b')
ax.set_xticks(list(x))
ax.set_xticklabels(speed_df['Model'], rotation=25, ha='right')
ax.set_ylabel('Time (seconds)')
ax.set_title('Fit + Predict Time by Model')
ax.legend()
plt.tight_layout()
plt.show()

print('\nNote: embedding encoding time is NOT included (done up front).')
print('Add encode time to get the true total for embedding models.')

## Section 4: Error Analysis

Where does the best classical model succeed but the best embedding model fail — and vice versa? These disagreements reveal what each representation captures or misses.

In [ ]:
classical_rows = [r for r in rows if r['Representation'] != 'Embeddings']
embedding_rows = [r for r in rows if r['Representation'] == 'Embeddings']

best_classical = max(classical_rows, key=lambda r: r['Macro F1'])
best_embedding = max(embedding_rows, key=lambda r: r['Macro F1'])

print(f"Best classical: {best_classical['Model']}  (F1={best_classical['Macro F1']})")
print(f"Best embedding: {best_embedding['Model']}  (F1={best_embedding['Macro F1']})")

preds_c = best_classical['_preds']
preds_e = best_embedding['_preds']
y_true  = test.target

c_right_e_wrong = np.where((preds_c == y_true) & (preds_e != y_true))[0]
e_right_c_wrong = np.where((preds_e == y_true) & (preds_c != y_true))[0]

print(f'\nClassical correct, Embedding wrong: {len(c_right_e_wrong)} cases')
print(f'Embedding correct, Classical wrong: {len(e_right_c_wrong)} cases')

In [ ]:
print('=== Cases where EMBEDDING wins but CLASSICAL fails ===\n')
for i in e_right_c_wrong[:2]:
    print(f"True:      {train.target_names[y_true[i]]}")
    print(f"Classical: {train.target_names[preds_c[i]]}  X")
    print(f"Embedding: {train.target_names[preds_e[i]]}  OK")
    print(f"Text: {test.data[i][:300]}")
    print('-' * 60)

print('\n=== Cases where CLASSICAL wins but EMBEDDING fails ===\n')
for i in c_right_e_wrong[:2]:
    print(f"True:      {train.target_names[y_true[i]]}")
    print(f"Classical: {train.target_names[preds_c[i]]}  OK")
    print(f"Embedding: {train.target_names[preds_e[i]]}  X")
    print(f"Text: {test.data[i][:300]}")
    print('-' * 60)

## Section 5: Per-Category Breakdown

In [ ]:
short_names = [c.split('.')[-1] for c in train.target_names]

print('=' * 55)
print(f"Classification Report -- {best_classical['Model']}")
print('=' * 55)
print(classification_report(y_true, preds_c, target_names=short_names))

print('=' * 55)
print(f"Classification Report -- {best_embedding['Model']}")
print('=' * 55)
print(classification_report(y_true, preds_e, target_names=short_names))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, preds, name in [
    (axes[0], preds_c, best_classical['Model']),
    (axes[1], preds_e, best_embedding['Model']),
]:
    cm = confusion_matrix(y_true, preds)
    sns.heatmap(cm, annot=True, fmt='d', square=True,
                xticklabels=short_names, yticklabels=short_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(name, fontsize=9)

plt.suptitle('Confusion Matrices: Best Classical vs. Best Embedding', fontsize=12)
plt.tight_layout()
plt.show()

## Section 6: Grid Search — TF-IDF + Logistic Regression

`GridSearchCV` automates hyperparameter search by exhaustively evaluating every combination in a parameter grid using **k-fold cross-validation**. The key difference from the model grid above:

| | Model Grid | Grid Search |
|---|---|---|
| Evaluation | Single train/test split | k-fold CV on training set |
| Parameter space | Hand-crafted combinations | Cartesian product of grid |
| Overfitting risk | Higher (one test set) | Lower (averaged over k folds) |

Below we search over TF-IDF and Logistic Regression hyperparameters simultaneously with 5-fold CV scored by macro F1.

In [ ]:
gs_pipe = Pipeline([
    ('vec', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

param_grid = {
    'vec__ngram_range' : [(1, 1), (1, 2)],
    'vec__max_features': [5000, 10000],
    'vec__sublinear_tf': [True, False],
    'clf__C'           : [0.1, 1.0, 10.0],
}

gs = GridSearchCV(gs_pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1)
gs.fit(train.data, train.target)

print(f'Best params:        {gs.best_params_}')
print(f'Best CV F1 (macro): {gs.best_score_:.4f}')
print()

gs_results = pd.DataFrame(gs.cv_results_)
cols = [
    'param_vec__ngram_range', 'param_vec__max_features',
    'param_vec__sublinear_tf', 'param_clf__C',
    'mean_test_score', 'std_test_score', 'rank_test_score',
]
print(gs_results[cols].sort_values('rank_test_score').head(10).to_string(index=False))

y_pred_gs = gs.best_estimator_.predict(test.data)
print('\nTest-set performance (best grid search model):')
print('=' * 60)
print(classification_report(test.target, y_pred_gs, target_names=train.target_names))

## Section 7: Feature Importance

Three complementary ways to understand what the model has learned:

| Method | Scope | Direction | Model |
|---|---|---|---|
| RF Feature Importance | Global | Unsigned | Random Forest |
| LR Coefficients | Per-class | Signed | Logistic Regression |
| SHAP | Local (per-doc) | Signed | Any linear model |

In [ ]:
# Pull TF-IDF + RandomForest from the results rows
rf_row        = next(r for r in rows if r['Representation'] == 'TF-IDF'
                     and 'RandomForest' in r['Classifier'])
rf_clf        = rf_row['_clf']
feature_names = tfidf.get_feature_names_out()
importances   = rf_clf.feature_importances_

top_n   = 20
top_idx = importances.argsort()[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(top_n), importances[top_idx], color='#5b8dd9')
ax.set_xticks(range(top_n))
ax.set_xticklabels(feature_names[top_idx], rotation=45, ha='right')
ax.set_ylabel('Feature Importance')
ax.set_title(f'Top {top_n} Words -- Random Forest (Global, Unsigned)')
plt.tight_layout()
plt.show()

In [ ]:
# Pull TF-IDF + LogisticReg from the results rows
lr_row      = next(r for r in rows if r['Representation'] == 'TF-IDF'
                   and 'LogisticReg' in r['Classifier'])
lr_clf      = lr_row['_clf']
lr_features = tfidf.get_feature_names_out()

n_cats  = len(train.target_names)
top_per = 10
fig, axes = plt.subplots(1, n_cats, figsize=(5 * n_cats, 5))

for ax, class_idx, cat in zip(axes, range(n_cats), train.target_names):
    coefs   = lr_clf.coef_[class_idx]
    top_pos = coefs.argsort()[::-1][:top_per]
    words   = lr_features[top_pos]
    scores  = coefs[top_pos]
    ax.barh(list(words)[::-1], list(scores)[::-1], color='#5bd95b')
    ax.set_title(cat.split('.')[-1], fontsize=10)
    ax.set_xlabel('LR Coefficient')

plt.suptitle('Top Words per Class -- LR Coefficients (Per-Class, Signed)', fontsize=13)
plt.tight_layout()
plt.show()

### SHAP -- SHapley Additive exPlanations

SHAP assigns each word a credit score for a **specific prediction**: positive values push toward the predicted class, negative values push away. This is fundamentally different from the global/unsigned methods above.

We use `shap.LinearExplainer` with a Logistic Regression fitted on `tfidf_shap` (5000 features — keeps the dense background dataset manageable at ~43 MB).

In [ ]:
# Fit a dedicated LR on tfidf_shap features for SHAP attribution
lr_explain_shap = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)
lr_explain_shap.fit(X_train_tfidf_shap, train.target)
shap_features = tfidf_shap.get_feature_names_out()

# LinearExplainer requires dense arrays
X_train_shap_dense = X_train_tfidf_shap.toarray()
X_test_shap_dense  = X_test_tfidf_shap[:200].toarray()

explainer   = shap.LinearExplainer(lr_explain_shap, X_train_shap_dense)
shap_values = explainer.shap_values(X_test_shap_dense)

print(f'Classes:              {len(shap_values)}')
print(f'SHAP shape per class: {shap_values[0].shape}')
print('One array per class; rows = documents, columns = vocabulary features.')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, cat in enumerate(train.target_names):
    plt.sca(axes[i])
    shap.summary_plot(
        shap_values[i],
        X_test_shap_dense,
        feature_names=shap_features,
        plot_type='bar',
        max_display=10,
        show=False,
    )
    axes[i].set_title(cat.split('.')[-1], fontsize=11)

plt.suptitle('SHAP Feature Importance per Class (Top 10 by Mean |SHAP|)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
doc_i    = 5
doc_text = test.data[doc_i]
true_cat = train.target_names[test.target[doc_i]]
pred_cat = train.target_names[lr_explain_shap.predict(X_test_tfidf_shap[doc_i])[0]]

print(f'True:      {true_cat}')
print(f'Predicted: {pred_cat}')
print()
print('Document excerpt:')
print(doc_text[:400])
print()

pred_class_idx  = lr_explain_shap.predict(X_test_tfidf_shap[doc_i])[0]
sv              = shap_values[pred_class_idx][doc_i]
top_contrib_idx = np.abs(sv).argsort()[::-1][:15]
contrib_words   = shap_features[top_contrib_idx]
contrib_vals    = sv[top_contrib_idx]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#5bd95b' if v > 0 else '#d95b5b' for v in contrib_vals[::-1]]
ax.barh(list(contrib_words[::-1]), list(contrib_vals[::-1]), color=colors)
ax.axvline(0, color='white', linewidth=0.5)
ax.set_xlabel('SHAP Value  (positive = pushes toward predicted class)')
ax.set_title(f"Local SHAP -- Doc {doc_i}  |  True: {true_cat.split('.')[-1]}  "
             f"|  Predicted: {pred_cat.split('.')[-1]}")
plt.tight_layout()
plt.show()

## Conclusion -- When to Use What

| Factor | Classical (CountVec/TF-IDF) | Semantic (Embeddings) |
|---|---|---|
| **Accuracy** | Good | Often better |
| **Speed** | Fast | Slower (inference) |
| **Interpretability** | High -- features are words | Low -- features are latent dims |
| **Data size** | Works well with small corpora | Shines on small datasets (pre-trained) |
| **Domain transfer** | Weak -- vocab must match | Strong -- pre-trained on broad text |
| **Memory** | Sparse (efficient) | Dense (larger) |

### Practical rules of thumb

- **Start with TF-IDF + Logistic Regression.** It's fast, interpretable, and surprisingly strong.
- **Use SHAP or LR coefficients** to understand what the classical model learned.
- **Upgrade to embeddings** when you need better generalization, have short texts, or the vocabulary varies a lot between train and test.
- **Don't use Random Forest on text** unless you have a specific reason -- it's slow and rarely beats LR on sparse features.

## Feature Importance: Method Comparison

| | RF Feature Importance | LR Coefficients | SHAP |
|---|---|---|---|
| Scope | Global (whole dataset) | Per-class | Local (per prediction) |
| Direction | No sign (always positive) | Signed | Signed |
| Model | Random Forest | Logistic Regression | Any linear model |
| Best for | Overall word importance | Which words define each class | Why one doc was predicted X |

**Next ->** Notebook 4 applies these techniques to real-world text corpora.